# 21 — Preliminary Submission Package Preparation

Mục tiêu: chuẩn bị trước các artifact có thể hoàn thiện **mà không cần biết interface TEST cuối của BTC**.

Notebook này tạo:
- `[Ten_Doi_Khong_Dau].pkl`
- `[Ten_Doi_Khong_Dau].JSON`
- report assets:
  - Loss Curve
  - Accuracy Curve
  - Confusion Matrix
  - ROC/AUC Curve
  - F1-score table
  - System pipeline diagram
  - Model architecture diagram
- checklist/preflight cho bộ hồ sơ vòng sơ loại

**Chưa tạo file `.keras` cuối cùng** ở notebook này. Core Keras ensemble đã PASS parity; wrapper input cuối sẽ chốt sau khi BTC xác nhận cách họ đưa ảnh vào `model.predict()`.

Không retune model, threshold hoặc ensemble weight trong notebook này.


In [ ]:
print('>>> CELL 1 START: config', flush=True)
from pathlib import Path
import json,pickle,shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix,roc_curve,roc_auc_score,f1_score

ROOT=Path(r'D:\medicine')

# EDIT THIS ONCE when team name is final.
TEAM_NAME='Btrick'

LOCK_PATH=ROOT/'cv_outputs'/'final_ensemble_lock'/'locked_pipeline.json'
HOLDOUT_JSON=ROOT/'holdout_outputs'/'holdout_evaluation.json'
HOLDOUT_PRED=ROOT/'holdout_outputs'/'holdout_predictions.csv'
KERAS_CORE=ROOT/'conversion_outputs'/'full_keras_ensemble'/'locked_densenet10_ensemble_core.keras'

PACKAGE=ROOT/'prelim_submission_prep'
ASSETS=PACKAGE/'report_assets'
PACKAGE.mkdir(parents=True,exist_ok=True)
ASSETS.mkdir(parents=True,exist_ok=True)

PKL_PATH=PACKAGE/f'{TEAM_NAME}.pkl'
JSON_PATH=PACKAGE/f'{TEAM_NAME}.JSON'

for p in [LOCK_PATH,HOLDOUT_JSON,HOLDOUT_PRED,KERAS_CORE]:
    assert p.exists(),f'Missing required source: {p}'

with open(LOCK_PATH,'r',encoding='utf-8') as f:
    lock=json.load(f)
with open(HOLDOUT_JSON,'r',encoding='utf-8') as f:
    holdout_result=json.load(f)

print('Team name placeholder:',TEAM_NAME,flush=True)
print('Locked threshold:',lock['threshold'],flush=True)
print('Core Keras:',KERAS_CORE,flush=True)


In [ ]:
print('>>> CELL 2 START: build model metadata .pkl', flush=True)

metadata={
    'competition':'Medical Image Classification Challenge 2026',
    'task':'Binary chest X-ray classification: Normal vs Pneumonia',
    'framework_final':'TensorFlow/Keras',
    'source_training_framework':'PyTorch/torchvision',
    'architecture':'DenseNet121 ensemble',
    'input_size':[224,224],
    'channels':3,
    'source_images':'grayscale',
    'preprocessing':[
        'convert input to grayscale',
        'replicate grayscale to 3 RGB channels',
        'pad to square with black border',
        'resize to 224x224 using bilinear interpolation',
        'ImageNet normalization mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]'
    ],
    'augmentation_training':[
        'RandomAffine degrees=5',
        'translation up to 2%',
        'scale 0.95-1.05',
        'ColorJitter brightness=0.08, contrast=0.08'
    ],
    'label_mapping':{
        'Normal':0,
        'Pneumonia':1
    },
    'output_order':['Normal','Pneumonia'],
    'pneumonia_threshold':float(lock['threshold']),
    'ensemble':[
        {
            'name':m['name'],
            'weight':float(m['weight']),
            'fold_models':5
        }
        for m in lock['models']
    ],
    'tta':False,
    'internal_oof_metrics':lock['oof_metrics'],
    'sealed_holdout_metrics':holdout_result['metrics'],
    'sealed_holdout_images':int(holdout_result['holdout_images']),
    'core_keras_file':KERAS_CORE.name,
    'notes':[
        'Model selection and threshold were locked before opening the sealed holdout.',
        'Holdout was used once for unbiased evaluation and not for retuning.',
        'Final BTC-facing raw-input wrapper is pending organizer input-interface confirmation.'
    ]
}

with open(PKL_PATH,'wb') as f:
    pickle.dump(metadata,f,protocol=pickle.HIGHEST_PROTOCOL)

print('Saved:',PKL_PATH,flush=True)
print('PKL keys:',list(metadata.keys()),flush=True)


In [ ]:
print('>>> CELL 3 START: build internal performance .JSON', flush=True)

performance_json={
    'competition':'Medical Image Classification Challenge 2026',
    'team_name_placeholder':TEAM_NAME,
    'metric_formula':'P = 0.30*Accuracy + 0.25*Sensitivity + 0.20*Specificity + 0.25*F1',
    'label_mapping':{'Normal':0,'Pneumonia':1},
    'locked_threshold':float(lock['threshold']),
    'oof_5fold':lock['oof_metrics'],
    'oof_fold_metrics':lock['fold_metrics'],
    'sealed_holdout':holdout_result['metrics'],
    'sealed_holdout_images':int(holdout_result['holdout_images']),
    'model':{
        'architecture':'DenseNet121',
        'ensemble':'2 seeds x 5 folds = 10 checkpoints',
        'weights':{
            m['name']:float(m['weight']) for m in lock['models']
        },
        'tta':False
    },
    'data_integrity':{
        'holdout_locked_before_evaluation':True,
        'holdout_retuning_performed':False,
        'external_test_used_for_tuning':False
    }
}

with open(JSON_PATH,'w',encoding='utf-8') as f:
    json.dump(performance_json,f,indent=2,ensure_ascii=False)

print('Saved:',JSON_PATH,flush=True)
print(json.dumps(performance_json['sealed_holdout'],indent=2),flush=True)


In [ ]:
print('>>> CELL 4 START: load training histories', flush=True)

history_specs=[
    ('seed2026',ROOT/'cv_outputs'/'densenet121_224_gray_b8a2'),
    ('seed3407',ROOT/'cv_outputs'/'densenet121_224_gray_b8a2_seed3407'),
]

histories=[]
for seed_name,d in history_specs:
    for fold in range(5):
        p=d/f'fold{fold}_history.csv'
        assert p.exists(),f'Missing history: {p}'
        h=pd.read_csv(p)
        h['seed']=seed_name
        h['fold_id']=fold
        histories.append(h)

hist=pd.concat(histories,ignore_index=True)

print('History rows:',len(hist),flush=True)
print('Columns:',hist.columns.tolist(),flush=True)
display(hist.head())


In [ ]:
print('>>> CELL 5 START: Loss Curve', flush=True)

plt.figure(figsize=(9,5.5))

for (seed,fold),g in hist.groupby(['seed','fold_id']):
    g=g.sort_values('epoch')
    plt.plot(g['epoch'],g['train_loss'],alpha=0.25,linewidth=1)

mean_loss=hist.groupby('epoch',as_index=False)['train_loss'].mean()
plt.plot(mean_loss['epoch'],mean_loss['train_loss'],linewidth=2.5,label='Mean across 10 runs')

plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('Training Loss Curve — DenseNet121 Ensemble')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

LOSS_PATH=ASSETS/'01_loss_curve.png'
plt.savefig(LOSS_PATH,dpi=220,bbox_inches='tight')
plt.show()

print('Saved:',LOSS_PATH,flush=True)


In [ ]:
print('>>> CELL 6 START: Accuracy Curve', flush=True)

plt.figure(figsize=(9,5.5))

for (seed,fold),g in hist.groupby(['seed','fold_id']):
    g=g.sort_values('epoch')
    plt.plot(g['epoch'],g['Accuracy'],alpha=0.25,linewidth=1)

mean_acc=hist.groupby('epoch',as_index=False)['Accuracy'].mean()
plt.plot(mean_acc['epoch'],mean_acc['Accuracy'],linewidth=2.5,label='Mean validation accuracy')

plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.title('Validation Accuracy Curve — DenseNet121 Ensemble')
plt.ylim(0.90,1.0)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

ACC_PATH=ASSETS/'02_accuracy_curve.png'
plt.savefig(ACC_PATH,dpi=220,bbox_inches='tight')
plt.show()

print('Saved:',ACC_PATH,flush=True)


In [ ]:
print('>>> CELL 7 START: holdout confusion matrix + ROC/AUC', flush=True)

pred_df=pd.read_csv(HOLDOUT_PRED)
required={'y_true','p_locked_ensemble','prediction'}
assert required.issubset(pred_df.columns),pred_df.columns.tolist()

y=pred_df['y_true'].astype(int).to_numpy()
p=pred_df['p_locked_ensemble'].astype(float).to_numpy()
pred=pred_df['prediction'].astype(int).to_numpy()

# Confusion matrix
cm=confusion_matrix(y,pred,labels=[0,1])

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.xticks([0,1],['Normal','Pneumonia'])
plt.yticks([0,1],['Normal','Pneumonia'])
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix — Sealed Holdout')

for i in range(2):
    for j in range(2):
        plt.text(j,i,str(cm[i,j]),ha='center',va='center')

plt.colorbar()
plt.tight_layout()
CM_PATH=ASSETS/'03_confusion_matrix.png'
plt.savefig(CM_PATH,dpi=220,bbox_inches='tight')
plt.show()

# ROC
fpr,tpr,_=roc_curve(y,p)
auc=roc_auc_score(y,p)

plt.figure(figsize=(6.5,5.5))
plt.plot(fpr,tpr,linewidth=2,label=f'ROC AUC = {auc:.6f}')
plt.plot([0,1],[0,1],'--',linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Sealed Holdout')
plt.grid(alpha=0.25)
plt.legend(loc='lower right')
plt.tight_layout()

ROC_PATH=ASSETS/'04_roc_auc_curve.png'
plt.savefig(ROC_PATH,dpi=220,bbox_inches='tight')
plt.show()

print('Saved:',CM_PATH,flush=True)
print('Saved:',ROC_PATH,flush=True)
print('AUC:',auc,flush=True)


In [ ]:
print('>>> CELL 8 START: F1-score table', flush=True)

rows=[]

oof=lock['oof_metrics']
rows.append({
    'Evaluation':'5-fold OOF locked ensemble',
    'Images':4362,
    'F1':float(oof['F1']),
    'Accuracy':float(oof['Accuracy']),
    'Sensitivity':float(oof['Sensitivity']),
    'Specificity':float(oof['Specificity']),
    'AUROC':float(oof['AUROC']),
    'P':float(oof['P'])
})

hm=holdout_result['metrics']
rows.append({
    'Evaluation':'One-time sealed holdout',
    'Images':int(holdout_result['holdout_images']),
    'F1':float(hm['F1']),
    'Accuracy':float(hm['Accuracy']),
    'Sensitivity':float(hm['Sensitivity']),
    'Specificity':float(hm['Specificity']),
    'AUROC':float(hm['AUROC']),
    'P':float(hm['P'])
})

for fm in lock['fold_metrics']:
    rows.append({
        'Evaluation':f"OOF Fold {fm['fold']}",
        'Images':'',
        'F1':float(fm['F1']),
        'Accuracy':float(fm['Accuracy']),
        'Sensitivity':float(fm['Sensitivity']),
        'Specificity':float(fm['Specificity']),
        'AUROC':float(fm['AUROC']),
        'P':float(fm['P'])
    })

f1_table=pd.DataFrame(rows)
F1_CSV=ASSETS/'05_f1_score_table.csv'
f1_table.to_csv(F1_CSV,index=False)

display(f1_table)
print('Saved:',F1_CSV,flush=True)


In [ ]:
print('>>> CELL 9 START: system pipeline diagram', flush=True)

fig,ax=plt.subplots(figsize=(13,3.4))
ax.axis('off')

labels=[
    'Chest X-ray\nInput',
    'Grayscale → RGB\nPad square → 224×224\nImageNet normalization',
    'Training augmentation\nAffine + brightness/contrast\n(train only)',
    'DenseNet121 ensemble\n2 seeds × 5 folds',
    'Weighted averaging\n0.70 / 0.30',
    'Threshold 0.576\nNormal / Pneumonia'
]

xs=np.linspace(0.06,0.94,len(labels))
for i,(x,label) in enumerate(zip(xs,labels)):
    ax.text(
        x,0.5,label,
        ha='center',va='center',
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.5',facecolor='white',edgecolor='black')
    )
    if i<len(labels)-1:
        ax.annotate(
            '',
            xy=(xs[i+1]-0.065,0.5),
            xytext=(x+0.065,0.5),
            xycoords=ax.transAxes,
            arrowprops=dict(arrowstyle='->',linewidth=1.4)
        )

ax.set_title('System Block Diagram — Medical Image Classification Pipeline',pad=20)
plt.tight_layout()

PIPE_PATH=ASSETS/'06_system_pipeline_diagram.png'
plt.savefig(PIPE_PATH,dpi=220,bbox_inches='tight')
plt.show()

print('Saved:',PIPE_PATH,flush=True)


In [ ]:
print('>>> CELL 10 START: model architecture diagram', flush=True)

fig,ax=plt.subplots(figsize=(12,5))
ax.axis('off')

labels=[
    'Input\n224×224×3',
    'Conv 7×7, s=2\nMaxPool',
    'Dense Block 1\n6 layers',
    'Transition 1',
    'Dense Block 2\n12 layers',
    'Transition 2',
    'Dense Block 3\n24 layers',
    'Transition 3',
    'Dense Block 4\n16 layers',
    'BN + ReLU\nGlobal Avg Pool',
    'FC 2 classes'
]

xs=np.linspace(0.04,0.96,len(labels))
y=0.62

for i,(x,label) in enumerate(zip(xs,labels)):
    ax.text(
        x,y,label,
        ha='center',va='center',
        transform=ax.transAxes,
        fontsize=8.5,
        bbox=dict(boxstyle='round,pad=0.35',facecolor='white',edgecolor='black')
    )
    if i<len(labels)-1:
        ax.annotate(
            '',
            xy=(xs[i+1]-0.035,y),
            xytext=(x+0.035,y),
            xycoords=ax.transAxes,
            arrowprops=dict(arrowstyle='->',linewidth=1.1)
        )

ax.text(
    0.5,0.23,
    '10 converted DenseNet121 models → 5-fold mean per seed → 0.70 × seed2026 + 0.30 × seed3407',
    ha='center',va='center',transform=ax.transAxes,
    bbox=dict(boxstyle='round,pad=0.5',facecolor='white',edgecolor='black')
)

ax.set_title('Model Architecture — DenseNet121 Multi-seed 5-fold Ensemble',pad=20)
plt.tight_layout()

ARCH_PATH=ASSETS/'07_model_architecture_diagram.png'
plt.savefig(ARCH_PATH,dpi=220,bbox_inches='tight')
plt.show()

print('Saved:',ARCH_PATH,flush=True)


In [ ]:
print('>>> CELL 11 START: submission preflight', flush=True)

core_size=KERAS_CORE.stat().st_size/(1024**2)

preflight=pd.DataFrame([
    {
        'Artifact':f'{TEAM_NAME}.keras',
        'Status':'WAITING_BTC_INPUT_INTERFACE',
        'Current source':str(KERAS_CORE),
        'Note':f'Core ensemble PASS parity; size={core_size:.2f} MiB'
    },
    {
        'Artifact':f'{TEAM_NAME}.pkl',
        'Status':'READY_DRAFT',
        'Current source':str(PKL_PATH),
        'Note':'Rename team placeholder after team name is final'
    },
    {
        'Artifact':f'{TEAM_NAME}.JSON',
        'Status':'READY_DRAFT',
        'Current source':str(JSON_PATH),
        'Note':'Internal OOF + sealed holdout metrics'
    },
    {
        'Artifact':f'{TEAM_NAME}.ipynb',
        'Status':'TO_BUILD_FINAL',
        'Current source':'notebooks 01-21',
        'Note':'Need one clean reproducible competition notebook'
    },
    {
        'Artifact':'Technical report',
        'Status':'ASSETS_READY',
        'Current source':str(ASSETS),
        'Note':'Loss, accuracy, CM, ROC/AUC, F1 table, pipeline, architecture'
    }
])

PREFLIGHT=PACKAGE/'submission_preflight.csv'
preflight.to_csv(PREFLIGHT,index=False)

display(preflight)
print('Saved:',PREFLIGHT,flush=True)


## Output sau notebook 21

Thư mục:

```text
D:\medicine\prelim_submission_prep\
```

sẽ có draft `.pkl`, `.JSON`, checklist, và toàn bộ report assets.

### Chưa làm ở bước này
- Không đổi threshold.
- Không dùng external test để tuning.
- Không đổi ensemble weight.
- Không đặt core `.keras` thành file nộp cuối cho tới khi BTC xác nhận input contract.
- Chưa tạo notebook nộp cuối; notebook đó sẽ gom pipeline thành một bản sạch, portable và khớp 100% với báo cáo.
